In [1]:
import numpy as np
from collections import Counter
from collections import defaultdict
from math import log
import pandas as pd
import xarray as xr
from itertools import product
import hexMinisom
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib import colormaps
from mpl_toolkits.axes_grid1 import make_axes_locatable
import seaborn as sns
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cartopy.feature as cf
import pickle
import itertools
import colorsys
import random
import datetime as dt

In [2]:
# =========================
# Save / Load SOM
# =========================
def save_som(som, fileName: str) -> None:
    """Save a SOM object to disk using pickle."""
    with open(fileName, "wb") as outfile:
        pickle.dump(som, outfile, protocol=pickle.HIGHEST_PROTOCOL)


def load_som(fileName: str):
    """Load a SOM object from disk using pickle."""
    with open(fileName, "rb") as infile:
        som = pickle.load(infile)
    return som


# =========================
# Colors
# =========================
def generate_distinct_colors(n: int):
    """Generate n visually distinct colors using evenly spaced HSV hues."""
    if n <= 0:
        return []

    colors = []
    for i in range(n):
        hue = i / n
        saturation = 1.0
        value = 1.0
        rgb = colorsys.hsv_to_rgb(hue, saturation, value)  # in [0,1]
        rgb255 = [int(x * 255) for x in rgb]
        hex_color = f"#{rgb255[0]:02x}{rgb255[1]:02x}{rgb255[2]:02x}"
        colors.append(hex_color)

    random.shuffle(colors)
    return colors


# =========================
# WR counts
# =========================
def get_WR_counts(
    l,
    return_percents: bool = False,
    indices=None,
    WR_labels=None,
    WR_labels_dict=None,
):
    """
    Count regimes for items in l, optionally restricted to a set of indices.

    Expects:
      - l: iterable of integer indices into WR_labels
      - WR_labels: list/array-like regime label per day/sample
      - WR_labels_dict: dict-like defining the full set of regimes (keys) and desired order

    Returns:
      dict[regime] -> count (or percent)
    """

    # Fallback to globals if not provided
    if WR_labels is None:
        WR_labels = globals().get("WR_labels", None)
    if WR_labels_dict is None:
        WR_labels_dict = globals().get("WR_labels_dict", None)

    if WR_labels is None or WR_labels_dict is None:
        raise ValueError("WR_labels and WR_labels_dict must be provided (or exist as globals).")

    regimes = list(WR_labels_dict.keys())  # stable order
    counts_dict = {r: 0 for r in regimes}

    if indices is None:
        selected = [WR_labels[i] for i in l]
    else:
        indices_set = set(indices)
        selected = [WR_labels[i] for i in l if i in indices_set]

    for r in selected:
        if r in counts_dict:
            counts_dict[r] += 1
        else:
            # In case an unexpected regime appears, keep it instead of crashing
            counts_dict[r] = counts_dict.get(r, 0) + 1

    if return_percents:
        total = sum(counts_dict.values())
        if total == 0:
            # no data for this node after filtering
            return {r: 0.0 for r in counts_dict}
        counts_dict = {r: (100.0 * c / total) for r, c in counts_dict.items()}

    return counts_dict


# =========================
# Hex plotting helpers
# =========================
def hex_heatmap(som, data, cmap="Blues", title="", cbLabel=""):
    """
    Plot a SOM heatmap in a hex layout.
    `data` is expected as dict[(i,j)] -> value for each node coordinate.
    """
    fig = plt.figure(figsize=(10, 10))
    ax = fig.add_subplot(111)
    ax.set_aspect("equal")

    cmap_obj = mpl.colormaps[cmap]

    weights = som.get_weights()
    xx, yy = som.get_euclidean_coordinates()

    # Determine min/max from provided data (fallback safe)
    values = list(data.values()) if len(data) > 0 else [0]
    maxCount = max(values)
    minCount = min(values)

    # Avoid division-by-zero / bad normalization
    if maxCount == minCount:
        # Make a tiny range so Normalize works and colors don't crash
        norm = mpl.colors.Normalize(vmin=minCount, vmax=minCount + 1e-12)
    else:
        norm = mpl.colors.Normalize(vmin=minCount, vmax=maxCount)

    # Loop neurons
    for i in range(weights.shape[0]):
        for j in range(weights.shape[1]):
            # Only use non-masked nodes
            if som._mask[i, j] == 0:
                value = data.get((i, j), 0)

                wy = yy[(j, i)] * np.sqrt(3) / 2
                face_color = cmap_obj(norm(value))

                hexagon = patches.RegularPolygon(
                    (xx[(j, i)], wy),
                    numVertices=6,
                    radius=0.85 / np.sqrt(3),
                    facecolor=face_color,
                    edgecolor="grey",
                )
                ax.add_patch(hexagon)

                # Text contrast heuristic using normalized intensity
                intensity = norm(value)
                textColor = "white" if intensity >= 0.75 else "black"

                ax.text(
                    xx[(j, i)],
                    wy - 0.07,
                    f"{value}",
                    ha="center",
                    va="center",
                    color=textColor,
                    fontsize=17,
                )

    # Align figure to show all hexagons
    ax.set_xlim(-1, weights.shape[0] - 0.5)
    ax.set_ylim(-1, (weights.shape[1] - 0.5) * np.sqrt(3) / 2)
    ax.axis("off")

    # Colorbar consistent with actual face colors
    sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap_obj)
    cb = fig.colorbar(
        sm,
        ax=ax,
        location="bottom",
        anchor=(0.56, 2.3),
        shrink=0.65,
        extend="both",
    )
    cb.set_label(cbLabel, fontsize=17)
    cb.ax.tick_params(labelsize=17)

    ax.set_title(title, fontsize=17, y=0.95, x=0.515)
    return fig


def hex_frequency_plot(som, winmap=None, dataarray=None):
    """
    Frequency per node: how many samples map to each BMU.
    Requires dataarray if winmap is None.
    """
    if winmap is None:
        if dataarray is None:
            dataarray = globals().get("dataarray", None)
        if dataarray is None:
            raise ValueError("dataarray must be provided if winmap is None (or exist as a global).")
        winmap = som.win_map(dataarray)

    data = {k: len(v) for k, v in winmap.items()}
    fig = hex_heatmap(som, data, cmap="Blues", title="(a) SOM Node Frequencies", cbLabel="Count")
    return fig


def hex_plot(som, projection=None, node_nums=None):
    """
    Create a matplotlib figure + dict of axes arranged into a hex-like SOM layout.

    Returns:
      fig, axs
      where axs[(y,x)] -> axis
    """
    n = som._num
    xy = (2 * n) - 1

    mask = som._mask
    node_indices_xy = np.ma.where(mask == False)
    node_indices = set(zip(node_indices_xy[0], node_indices_xy[1]))

    if node_nums is None:
        node_nums = globals().get("node_nums", None)

    # Grid dimensions
    # Rows are driven by x*3 (+ offset for odd y), cols driven by y*2
    grid_rows = xy * 3 + 2
    grid_cols = xy * 2 + 2

    fig = plt.figure(figsize=(grid_rows, grid_cols))
    axs = {}

    # Loop over a square and place only unmasked nodes
    for x, y in product(range(xy), range(xy)):
        # Keep your original orientation: check (y,x) membership
        if (y, x) in node_indices:
            # Row placement: x controls vertical stacking
            base_row = x * 3
            row_offset = 1 if (y % 2 != 0) else 0
            cur_row = base_row + row_offset

            # Column placement: reverse y so it visually matches your prior layout
            cur_col = (xy - 1 - y) * 2

            ax = plt.subplot2grid(
                (grid_rows, grid_cols),
                (cur_row, cur_col),
                rowspan=2,
                colspan=2,
                projection=projection,
            )

            if node_nums is not None and (y, x) in node_nums:
                ax.set_title(node_nums[(y, x)])

            if projection is not None:
                # keep your original map bounds
                ax.set_extent([-180, -30, 20, 80], crs=projection)

            axs[(y, x)] = ax

    return fig, axs


# =========================
# Longest consecutive run
# =========================
def find_longest_consecutive_index(arr):
    """Return the start index of the longest consecutive run of identical values."""
    if arr is None or len(arr) == 0:
        return -1

    max_len = 0
    start_index = -1
    current_len = 1
    current_start = 0

    for i in range(1, len(arr)):
        if arr[i] == arr[i - 1]:
            current_len += 1
        else:
            if current_len > max_len:
                max_len = current_len
                start_index = current_start
            current_len = 1
            current_start = i

    # Check final run
    if current_len > max_len:
        start_index = current_start

    return start_index


# =========================
# Trend significance
# =========================
def trend_significance(group, alpha=0.05, iters=10000, rng=None):
    """
    Bootstrap-style significance bounds for a linear trend slope.

    group: pandas Series-like, uses group.index as x values
    """
    if rng is None:
        rng = random

    g = list(group)
    if len(g) == 0:
        return (np.nan, np.nan)

    slopes = []

    for _ in range(iters):
        # resample with replacement
        samples = [rng.choice(g) for _ in range(len(g))]
        m = np.polyfit(group.index, samples, 1)[0]
        slopes.append(m)

    lower = np.percentile(slopes, (alpha / 2) * 100)
    upper = np.percentile(slopes, (1 - alpha / 2) * 100)
    return lower, upper


# =========================
# SOM class distribution metrics
# =========================
def compute_class_spread(winmap, y, n_classes):
    """
    For each class:
      - Get BMU coordinates of all samples
      - Compute centroid
      - Compute average Euclidean distance to centroid
    """
    class_spread = {}

    for cls in range(n_classes):
        bmu_coords = []

        for bmu, indices in winmap.items():
            for i in indices:
                if y[i] == cls:
                    bmu_coords.append(np.array(bmu))

        if not bmu_coords:
            class_spread[cls] = 0.0
            continue

        bmu_coords = np.array(bmu_coords)
        centroid = bmu_coords.mean(axis=0)
        distances = np.linalg.norm(bmu_coords - centroid, axis=1)
        class_spread[cls] = float(np.mean(distances))

    return class_spread


def compute_class_coverage(winmap, y, n_classes):
    """For each class, count how many unique SOM nodes are activated."""
    class_nodes = defaultdict(set)

    for bmu, indices in winmap.items():
        for i in indices:
            cls = y[i]
            class_nodes[cls].add(bmu)

    return {cls: len(class_nodes[cls]) for cls in range(n_classes)}


def compute_class_entropy(winmap, y, n_classes):
    """
    Entropy of SOM node distribution for each class.
    Entropy reflects how spread out each class is across the SOM.
    """
    class_entropy = {}

    for cls in range(n_classes):
        node_counts = []
        total = 0

        for bmu, indices in winmap.items():
            count = sum(1 for i in indices if y[i] == cls)
            if count > 0:
                node_counts.append(count)
                total += count

        if total == 0:
            class_entropy[cls] = 0.0
            continue

        probs = [count / total for count in node_counts]
        ent = -sum(p * log(p) for p in probs if p > 0)
        class_entropy[cls] = float(ent)

    return class_entropy


def compute_topographic_class_purity(winmap, y, n_classes):
    """
    Average node purity for each class.
    Purity here = proportion of cls in each node, averaged over nodes where cls appears.
    """
    class_purity = {}

    for cls in range(n_classes):
        purities = []

        for bmu, indices in winmap.items():
            labels = [y[i] for i in indices]
            if cls in labels:
                counts = Counter(labels)
                node_purity = counts[cls] / len(labels)
                purities.append(node_purity)

        class_purity[cls] = float(sum(purities) / len(purities)) if purities else 0.0

    return class_purity


def check_node_activation_by_class(winmap, y):
    """Print how many SOM nodes each class appears in."""
    class_node_count = defaultdict(set)

    for bmu, indices in winmap.items():
        labels_in_node = set(y[i] for i in indices)
        for label in labels_in_node:
            class_node_count[label].add(bmu)

    for cls in sorted(class_node_count.keys()):
        print(f"Class {cls} appears in {len(class_node_count[cls])} SOM nodes")


def compute_weighted_coverage(winmap, y, n_classes, method="effective"):
    """
    Weighted coverage of classes over SOM nodes.

    method:
      - 'effective': exp(entropy) => effective number of nodes
      - 'simpson': 1 / sum(p^2) => Simpson effective number of nodes

    Returns:
      dict[class] -> weighted coverage score (higher = more spread out)
    """
    class_node_counts = {cls: defaultdict(int) for cls in range(n_classes)}

    for bmu, indices in winmap.items():
        for i in indices:
            cls = y[i]
            class_node_counts[cls][bmu] += 1

    coverage = {}
    for cls in range(n_classes):
        counts = np.array(list(class_node_counts[cls].values()), dtype=float)
        if counts.size == 0:
            coverage[cls] = 0.0
            continue

        p = counts / counts.sum()

        if method == "effective":
            entropy = -np.sum(p * np.log(p + 1e-12))
            coverage[cls] = float(np.exp(entropy))
        elif method == "simpson":
            coverage[cls] = float(1.0 / np.sum(p ** 2))
        else:
            raise ValueError("method must be 'effective' or 'simpson'")

    return coverage

In [3]:
dataset = xr.open_dataarray('/glade/work/molina/DATA/Z500Anoms_ERA5.nc')

latSlice = slice(20, 80) #20N, 80N
lonSlice = slice(180, 330) #180W, 30W
dataarray = dataset.sel(lat=latSlice, lon=lonSlice)
dataarray = dataarray.stack(latlon=['lat', 'lon']).values

# Seasonal breakdown of the data
DJF = dataset.time.dt.month.isin([12, 1, 2])
DJF_idxs = np.array(DJF).nonzero()[0]
MAM = dataset.time.dt.month.isin([3, 4, 5])
MAM_idxs = np.array(MAM).nonzero()[0]
JJA = dataset.time.dt.month.isin([6, 7, 8])
JJA_idxs = np.array(JJA).nonzero()[0]
SON = dataset.time.dt.month.isin([9, 10, 11])
SON_idxs = np.array(SON).nonzero()[0]

print(dataarray.shape)

(30660, 9211)


In [4]:
som = load_som('SOM40.p')

n = som._num
xy = hexMinisom.xy_using_n(n)

mask = som._mask
node_indices_xy = np.ma.where(mask == False)
node_indices = list(zip(node_indices_xy[0], node_indices_xy[1]))
all_nodes = product(range(xy), range(xy))

inputLength = dataarray.shape[1]

winmap = som.win_map(dataarray, return_indices=True)
# Seasonal breakdown for winmap
DJF_winmap = {}
MAM_winmap = {}
JJA_winmap = {}
SON_winmap = {}

# Loop through each node
for k, v in winmap.items():
    
    # Keep only the days that are in the given season
    DJF_winmap[k] = [i for i in v if i in DJF_idxs]
    MAM_winmap[k] = [i for i in v if i in MAM_idxs]
    JJA_winmap[k] = [i for i in v if i in JJA_idxs]
    SON_winmap[k] = [i for i in v if i in SON_idxs]

w = som._weights
minimum_weight = -np.max(np.abs(w))
maximum_weight = np.max(np.abs(w))

# Calculate the node number for each coordinate
node_nums = {}
n = 1
for i in range(mask.shape[0])[::-1]:
    for j in range(mask.shape[1]):
        # only use non masked nodes
        if som._mask[i, j] == 0:
            node_nums[(i, j)] = n
            n += 1

color_list = generate_distinct_colors(len(node_indices))

In [5]:
# Import the regime labels
WR_labels_df = pd.read_csv('data/df_labels_nocorrfilt_ERA5.csv')
WR_labels_df.rename(columns={'Unnamed: 0': 'date'}, inplace=True)
WR_labels_df['date'] = pd.to_datetime(WR_labels_df['date'], format='%Y-%m-%d')
WR_labels_dict = {
    0: 'Greenland High', 1: 'Pacific Trough', 2: 'Pacific Ridge', 
    3: 'Alaskan Ridge', 4: 'North American\nAtlantic Ridge', 5: 'No WR'
}
WR_labels = np.array(WR_labels_df['WR'])

WRs_by_node = {k: get_WR_counts(v) for k, v in winmap.items()}
WRs_percents = {k: get_WR_counts(v, return_percents=True) for k, v in winmap.items()}

print(WR_labels_df)

# Calculate the 90th percentile of the distances and only keep data less than that
percentile90 = np.percentile(WR_labels_df['distances'], 90)
lt90 = (np.array(WR_labels_df['distances']) < percentile90).nonzero()[0]
WRs_lt90 = {k: get_WR_counts(v, True, lt90) for k, v in winmap.items()}

# Calculate the variances of the distances for each WR
WR_indices = {i: (WR_labels == i).nonzero()[0] for i in np.unique(WR_labels)}
for WR, idxs in WR_indices.items():
    variance = np.var(WR_labels_df['distances'].iloc[idxs])
    
# Get the WR counts for each specific season
WRs_DJF = {k: get_WR_counts(v, True, DJF_idxs) for k, v in winmap.items()}
WRs_MAM = {k: get_WR_counts(v, True, MAM_idxs) for k, v in winmap.items()}
WRs_JJA = {k: get_WR_counts(v, True, JJA_idxs) for k, v in winmap.items()}
WRs_SON = {k: get_WR_counts(v, True, SON_idxs) for k, v in winmap.items()}

            date  WR  distances      corr
0     1940-01-01   0   2.463938  0.518457
1     1940-01-02   0   2.662645  0.565398
2     1940-01-03   0   2.916932  0.552532
3     1940-01-04   0   3.122750  0.495652
4     1940-01-05   0   3.302769  0.394692
...          ...  ..        ...       ...
30655 2023-12-27   1   2.823370  0.857459
30656 2023-12-28   1   2.687266  0.837698
30657 2023-12-29   1   2.394127  0.781903
30658 2023-12-30   1   2.253624  0.655454
30659 2023-12-31   0   2.468174  0.359541

[30660 rows x 4 columns]


In [6]:
names_to_fill = []

for i in WR_labels_df['WR'].values:
    
    names_to_fill.append(WR_labels_dict[i])

array_to_fill = np.ones(len(WR_labels_df), dtype=int) * -9999

for i in range(0, len(node_indices)):
    
    for j in winmap[node_indices[i]]:
        
        array_to_fill[j] = node_nums[node_indices[i]]

WR_labels_df['node'] = array_to_fill
WR_labels_df['WR_name'] = names_to_fill

diversity_df = pd.concat([WR_labels_df, pd.DataFrame(dataarray)], axis=1)

# Extract feature columns — these are the SOM inputs
X = diversity_df.iloc[:, 6:]

# Extract class labels (e.g. WR categories)
y = diversity_df['WR'].values 

In [7]:
# total samples per regime
print(np.unique(y, return_counts=True))
print(len(y))

tmp_map = winmap

spread_scores = compute_class_spread(tmp_map, y, n_classes=6)
coverage_scores = compute_class_coverage(tmp_map, y, n_classes=6)
entropy_scores = compute_class_entropy(tmp_map, y, n_classes=6)
purity_scores = compute_topographic_class_purity(tmp_map, y, n_classes=6)
wcoverage_effective = compute_weighted_coverage(tmp_map, y, n_classes=6, method='effective')
wcoverage_simpson = compute_weighted_coverage(tmp_map, y, n_classes=6, method='simpson')

#for cls in range(6):
#    print(
#        f"Spread = {
#        spread_scores[cls]:.3f}, Coverage = {
#        coverage_scores[cls]}, Entropy = {
#        entropy_scores[cls]:.3f}, Purity = {
#        purity_scores[cls]:.3f}, Eff. Coverage = {
#        wcoverage_effective[cls]:.3f}, Simpson = {
#        wcoverage_simpson[cls]:.2f}: {WR_labels_dict[cls]}"
#    )

for cls in range(6):
    print(
        f"Coverage = {
        coverage_scores[cls]}, Simpson = {
        wcoverage_simpson[cls]:.2f}: {WR_labels_dict[cls]}"
    )

(array([0, 1, 2, 3, 4, 5]), array([5217, 6164, 5943, 4279, 5821, 3236]))
30660
Coverage = 36, Simpson = 10.57: Greenland High
Coverage = 37, Simpson = 21.48: Pacific Trough
Coverage = 37, Simpson = 13.22: Pacific Ridge
Coverage = 37, Simpson = 12.38: Alaskan Ridge
Coverage = 37, Simpson = 30.28: North American
Atlantic Ridge
Coverage = 37, Simpson = 30.41: No WR


In [8]:
# total samples per regime
print(np.unique(y[DJF_idxs], return_counts=True))
print(len(y[DJF_idxs]))

tmp_map = DJF_winmap

spread_scores = compute_class_spread(tmp_map, y, n_classes=6)
coverage_scores = compute_class_coverage(tmp_map, y, n_classes=6)
entropy_scores = compute_class_entropy(tmp_map, y, n_classes=6)
purity_scores = compute_topographic_class_purity(tmp_map, y, n_classes=6)
wcoverage_effective = compute_weighted_coverage(tmp_map, y, n_classes=6, method='effective')
wcoverage_simpson = compute_weighted_coverage(tmp_map, y, n_classes=6, method='simpson')

#for cls in range(6):
#    print(
#        f"Spread = {
#        spread_scores[cls]:.3f}, Coverage = {
#        coverage_scores[cls]}, Entropy = {
#        entropy_scores[cls]:.3f}, Purity = {
#        purity_scores[cls]:.3f}, Eff. Coverage = {
#        wcoverage_effective[cls]:.3f}, Simpson = {
#        wcoverage_simpson[cls]:.2f}: {WR_labels_dict[cls]}"
#    )

for cls in range(6):
    print(
        f"Coverage = {
        coverage_scores[cls]}, Simpson = {
        wcoverage_simpson[cls]:.2f}: {WR_labels_dict[cls]}"
    )

(array([0, 1, 2, 3, 4, 5]), array([1186, 1636, 1748,  889, 1352,  749]))
7560
Coverage = 31, Simpson = 8.60: Greenland High
Coverage = 37, Simpson = 18.06: Pacific Trough
Coverage = 32, Simpson = 11.53: Pacific Ridge
Coverage = 34, Simpson = 8.69: Alaskan Ridge
Coverage = 37, Simpson = 27.84: North American
Atlantic Ridge
Coverage = 37, Simpson = 26.17: No WR


In [9]:
# total samples per regime
print(np.unique(y[MAM_idxs], return_counts=True))
print(len(y[MAM_idxs]))

tmp_map = MAM_winmap

spread_scores = compute_class_spread(tmp_map, y, n_classes=6)
coverage_scores = compute_class_coverage(tmp_map, y, n_classes=6)
entropy_scores = compute_class_entropy(tmp_map, y, n_classes=6)
purity_scores = compute_topographic_class_purity(tmp_map, y, n_classes=6)
wcoverage_effective = compute_weighted_coverage(tmp_map, y, n_classes=6, method='effective')
wcoverage_simpson = compute_weighted_coverage(tmp_map, y, n_classes=6, method='simpson')

#for cls in range(6):
#    print(
#        f"Spread = {
#        spread_scores[cls]:.3f}, Coverage = {
#        coverage_scores[cls]}, Entropy = {
#        entropy_scores[cls]:.3f}, Purity = {
#        purity_scores[cls]:.3f}, Eff. Coverage = {
#        wcoverage_effective[cls]:.3f}, Simpson = {
#        wcoverage_simpson[cls]:.2f}: {WR_labels_dict[cls]}"
#    )

for cls in range(6):
    print(
        f"Coverage = {
        coverage_scores[cls]}, Simpson = {
        wcoverage_simpson[cls]:.2f}: {WR_labels_dict[cls]}"
    )

(array([0, 1, 2, 3, 4, 5]), array([1386, 1553, 1538,  999, 1485,  767]))
7728
Coverage = 32, Simpson = 9.11: Greenland High
Coverage = 37, Simpson = 23.05: Pacific Trough
Coverage = 36, Simpson = 11.98: Pacific Ridge
Coverage = 37, Simpson = 11.01: Alaskan Ridge
Coverage = 37, Simpson = 26.55: North American
Atlantic Ridge
Coverage = 36, Simpson = 29.22: No WR


In [10]:
# total samples per regime
print(np.unique(y[JJA_idxs], return_counts=True))
print(len(y[JJA_idxs]))

tmp_map = JJA_winmap

spread_scores = compute_class_spread(tmp_map, y, n_classes=6)
coverage_scores = compute_class_coverage(tmp_map, y, n_classes=6)
entropy_scores = compute_class_entropy(tmp_map, y, n_classes=6)
purity_scores = compute_topographic_class_purity(tmp_map, y, n_classes=6)
wcoverage_effective = compute_weighted_coverage(tmp_map, y, n_classes=6, method='effective')
wcoverage_simpson = compute_weighted_coverage(tmp_map, y, n_classes=6, method='simpson')

#for cls in range(6):
#    print(
#        f"Spread = {
#        spread_scores[cls]:.3f}, Coverage = {
#        coverage_scores[cls]}, Entropy = {
#        entropy_scores[cls]:.3f}, Purity = {
#        purity_scores[cls]:.3f}, Eff. Coverage = {
#        wcoverage_effective[cls]:.3f}, Simpson = {
#        wcoverage_simpson[cls]:.2f}: {WR_labels_dict[cls]}"
#    )

for cls in range(6):
    print(
        f"Coverage = {
        coverage_scores[cls]}, Simpson = {
        wcoverage_simpson[cls]:.2f}: {WR_labels_dict[cls]}"
    )

(array([0, 1, 2, 3, 4, 5]), array([1647, 1335, 1060, 1410, 1406,  870]))
7728
Coverage = 36, Simpson = 11.24: Greenland High
Coverage = 37, Simpson = 23.61: Pacific Trough
Coverage = 37, Simpson = 16.04: Pacific Ridge
Coverage = 37, Simpson = 16.08: Alaskan Ridge
Coverage = 37, Simpson = 28.24: North American
Atlantic Ridge
Coverage = 37, Simpson = 30.51: No WR


In [11]:
# total samples per regime
print(np.unique(y[SON_idxs], return_counts=True))
print(len(y[SON_idxs]))

tmp_map = SON_winmap

spread_scores = compute_class_spread(tmp_map, y, n_classes=6)
coverage_scores = compute_class_coverage(tmp_map, y, n_classes=6)
entropy_scores = compute_class_entropy(tmp_map, y, n_classes=6)
purity_scores = compute_topographic_class_purity(tmp_map, y, n_classes=6)
wcoverage_effective = compute_weighted_coverage(tmp_map, y, n_classes=6, method='effective')
wcoverage_simpson = compute_weighted_coverage(tmp_map, y, n_classes=6, method='simpson')

#for cls in range(6):
#    print(
#        f"Spread = {
#        spread_scores[cls]:.3f}, Coverage = {
#        coverage_scores[cls]}, Entropy = {
#        entropy_scores[cls]:.3f}, Purity = {
#        purity_scores[cls]:.3f}, Eff. Coverage = {
#        wcoverage_effective[cls]:.3f}, Simpson = {
#        wcoverage_simpson[cls]:.2f}: {WR_labels_dict[cls]}"
#    )

for cls in range(6):
    print(
        f"Coverage = {
        coverage_scores[cls]}, Simpson = {
        wcoverage_simpson[cls]:.2f}: {WR_labels_dict[cls]}"
    )

(array([0, 1, 2, 3, 4, 5]), array([ 998, 1640, 1597,  981, 1578,  850]))
7644
Coverage = 29, Simpson = 12.40: Greenland High
Coverage = 37, Simpson = 19.93: Pacific Trough
Coverage = 35, Simpson = 13.01: Pacific Ridge
Coverage = 37, Simpson = 10.30: Alaskan Ridge
Coverage = 37, Simpson = 30.00: North American
Atlantic Ridge
Coverage = 37, Simpson = 29.01: No WR
